###  炭素構造の階層クラスタリング

**データ取得から可視化**

In [ ]:
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd
import matplotlib.pylab as plt
%matplotlib inline

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 10)


In [ ]:
def df_avarag_atom(df0):
    """原子平均をして各結晶の説明変数とする。

    Args:
        df0 (pd.DataFrame): data with atomic explanatory variables

    Returns:
        pd.DataFrame: data with cell explanatory variables
    """

    Xlist = []
    for key in df0.index.levels[0]:

        X = df0.loc[key].values
        natom = X.shape[0]
        x = X.sum(axis=0)/natom

        xx = [key]
        xx.extend(x)

        Xlist.append(xx)

    columns = ["key"]
    columns.extend(df0.columns)
    df = pd.DataFrame(Xlist, columns=columns)
    return df.set_index(["key"])


def select_structure(df):
    """select structures

    Args:
        df (pd.DataFrame): data

    Returns:
        dp.DataFrame: selected data
    """
    keylist = []
    for key in df.index:
        s = key.split("-")
        dim = s[0]
        i = int(s[1])
        if (dim == "3D" or dim == "2D") and i <= 5:
            keylist.append(key)
    return df.loc[keylist]


クラスタリングは以下のように行えます。

In [ ]:
# データ取得（原子説明変数の読み込み）
g_df0 = pd.read_csv("../data_calculated/Carbon8_descriptor.csv", index_col=[0,1])
g_descriptor_names = g_df0.columns
g_df0

In [ ]:
# データ加工

# 説明変数生成
# 原子説明変数から結晶説明変数への変換
g_df_large = df_avarag_atom(g_df0)
g_df = select_structure(g_df_large)
g_label = list(g_df.index)
g_Xraw = g_df.values

# データ規格化
g_scaler = StandardScaler()
g_scaler.fit(g_Xraw)
g_X = g_scaler.transform(g_Xraw)
g_labels = list(g_df.index)

可視化を含めて解析していきます。
二次元で可視化するためにPCAにより次元圧縮を行っています。
これも次元圧縮の用途の一つです。

別のクラスタリング手法である
階層クラスタリングも行うことができます。

これは各サンプル点間の距離を計算し、最短の距離にある点を順につなげていくことで樹形図を作成する手法です。
距離の計算手法やグループ間の距離の計算手法がそれぞれ幾つか存在し、目的に応じて最適な手法を選択します。

In [ ]:
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.stats import pearsonr
import numpy as np
from scipy.spatial.distance import squareform
import seaborn as sns

def make_dendrogram(X, labels, metric='euclidean',figsize=(10,10), save_fig: bool=True):
    """階層クラスタリングを行う。

    Args:
        X (np.ndarray): normalized explanatory variables
        labels ([str]): 説明変数名
        metric (str, optional): metric名. Defaults to 'euclidean'.
        figsize (tuple, optional): 図のサイズ. Defaults to (10,10).

    Returns:
        nd.pdarray: pairdistance.
        dendrogram: denddrogram instance
    """
    print("X.shape",X.shape)
    print("labels", len(labels), labels)
    if True:
        if metric=="minus_abs_pearson":
            print(" metric", metric)
            df_tmp = pd.DataFrame(X)
            if True:
                corr = 1- np.abs(df_tmp.T.corr()) # DataFrame can calculate all the pearson's correlations
            else:
                corr = []
                for x1 in X:
                    corr1 = []
                    for x2 in X:
                        corr1.append(1.0-np.abs(pearsonr(x1,x2)[0]))
                    corr.append(corr1)
                corr = np.array(corr)
                for i in range(corr.shape[0]): # set the diagonal parts explicitly zero because of numerical precision.
                    corr[i,i] = 0.0
            print("corr",corr)
            print("corr.shape", corr.shape)
            pairdistance = squareform(corr)
            print("pairdistance", pairdistance.shape)
        else:
            print("metric",metric)
            pairdistance = pdist(X, metric=metric)  # calculate pair distance
        Z = linkage(pairdistance) # pairistance is 1D array
    else:
        Z = linkage(X,metric=metric) # 2D array
        
    fig, ax= plt.subplots(figsize=figsize)
    tree = dendrogram(Z, labels=labels, orientation="left", ax=ax)
    ax.invert_yaxis()
    fig.tight_layout()
    if save_fig:
        fig.savefig("image_executed/carbon8_dendrogram.png")
    fig.show()
    return pairdistance, tree

def make_pairdistance_matrix(pairdistance, labels, tree, figsize=(10,10)):
    pairdistance_matrix = squareform(pairdistance)
    df_pdistmatrix = pd.DataFrame(pairdistance_matrix, index=labels, columns=labels)
    df_pdistmatrix
    
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(df_pdistmatrix.loc[tree["ivl"],tree["ivl"]], ax=ax)
    # ax.invert_yaxis()

In [ ]:
g_pairdistance, g_tree = make_dendrogram(g_X, g_labels,figsize=(5,5), save_fig=False)
make_pairdistance_matrix(g_pairdistance, g_labels, g_tree)

In [ ]:
g_figsize=(5,5)
g_pairdistance, g_tree = make_dendrogram(g_X.T, g_descriptor_names, metric="minus_abs_pearson", 
                                         figsize=g_figsize, save_fig=True)
make_pairdistance_matrix(g_pairdistance, g_descriptor_names, g_tree, figsize=g_figsize)

seabornライブラリを用いるとheatmapと樹形図を組み合わせた図を簡単に書くこともできます。

In [ ]:
from scipy.spatial.distance import cdist
import seaborn as sns
    
g = sns.clustermap(g_df, metric="euclidean", figsize=(10,10) ) # can't use ax=